# Approximation algorithms for bin packing (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/bin-packing-approximation_python.ipynb)

Executable notebook of the chapter “Approximation algorithms for bin packing” (part “Approximation algorithms” of the course *Combinatorial Optimization*): it contains the four approximation algorithms of the chapter for the bin packing problem: Next Fit, First Fit, Best Fit and Worst Fit, written in Python and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/approximation-algorithms/bin-packing-approximation/). The same notebook is also available in [C++](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/bin-packing-approximation_cpp.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`: the cells must be run in order, since each one uses the functions defined above it. Nothing to install: only the Python standard library is used.

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Next Fit

Only the last opened bin is considered: if the item does not fit, a new bin is opened.

In [1]:
# Approximation algorithms for bin packing: items and bins are indexed from 0.
# Every function returns the pair (nb, bin): the number of bins used and the list bin,
# where bin[i] is the bin assigned to item i (the C++ version fills an output array).

def next_fit(n, c, w):
    """Next Fit: items are processed in the input order; it is assumed that n >= 1.
    Returns (nb, bin): the number of bins used and the bin assigned to each item."""
    bin = [0] * n
    nb = 1   # current bin (number of open bins)
    rc = c   # residual capacity of the current bin

    for i in range(n):
        # if item i does not fit into the current bin, open a new bin
        if w[i] > rc:
            nb += 1  # move to the next bin
            rc = c   # reset the residual capacity

        bin[i] = nb - 1  # pack item i into the current bin
        rc -= w[i]       # update the residual capacity

    return nb, bin

## First Fit

The item goes into the first opened bin (in opening order) with enough residual capacity; if there is none, a new bin is opened.

In [2]:
def first_fit(n, c, w):
    """First Fit: items are processed in the input order; it is assumed that n >= 1.
    Returns (nb, bin): the number of bins used and the bin assigned to each item."""
    bin = [0] * n
    nb = 1          # number of currently open bins
    rc = [0] * n    # residual capacities of bins (at most n bins)
    rc[0] = c       # residual capacity of the first bin

    for i in range(n):
        s = -1      # selected bin for item i (-1 means: not found yet)

        # scan bins in the order they were opened and pick the first that fits
        for b in range(nb):
            if w[i] <= rc[b]:
                s = b  # selected bin
                break

        # if no open bin can accommodate item i, open a new bin
        if s == -1:
            rc[nb] = c  # initialize residual capacity of the new bin
            s = nb      # select the new bin
            nb += 1     # increase the number of open bins

        bin[i] = s      # pack item i into the selected bin
        rc[s] -= w[i]   # update the residual capacity

    return nb, bin

## Best Fit

The item goes into the opened bin with the smallest residual capacity among those in which it fits.

In [3]:
def best_fit(n, c, w):
    """Best Fit: items are processed in the input order; it is assumed that n >= 1.
    Returns (nb, bin): the number of bins used and the bin assigned to each item."""
    bin = [0] * n
    nb = 1          # number of currently open bins
    rc = [0] * n    # residual capacities of bins (at most n bins)
    rc[0] = c       # residual capacity of the first bin

    for i in range(n):
        s = -1          # selected bin for item i (-1 means: not found yet)
        minrc = c + 1   # smallest residual capacity found so far

        # among the bins where item i fits, pick the one with the smallest residual capacity
        for b in range(nb):
            if w[i] <= rc[b] and rc[b] < minrc:
                minrc = rc[b]
                s = b  # selected bin

        # if no open bin can accommodate item i, open a new bin
        if s == -1:
            rc[nb] = c  # initialize residual capacity of the new bin
            s = nb      # select the new bin
            nb += 1     # increase the number of open bins

        bin[i] = s      # pack item i into the selected bin
        rc[s] -= w[i]   # update the residual capacity

    return nb, bin

## Worst Fit

The item goes into the opened bin with the largest residual capacity, if it fits there.

In [4]:
def worst_fit(n, c, w):
    """Worst Fit: items are processed in the input order; it is assumed that n >= 1.
    Returns (nb, bin): the number of bins used and the bin assigned to each item."""
    bin = [0] * n
    nb = 1          # number of currently open bins
    rc = [0] * n    # residual capacities of bins (at most n bins)
    rc[0] = c       # residual capacity of the first bin

    for i in range(n):
        s = -1      # selected bin for item i (-1 means: not found yet)
        maxrc = -1  # largest residual capacity found so far

        # among the bins where item i fits, pick the one with the largest residual capacity
        for b in range(nb):
            if w[i] <= rc[b] and rc[b] > maxrc:
                maxrc = rc[b]
                s = b  # selected bin

        # if no open bin can accommodate item i, open a new bin
        if s == -1:
            rc[nb] = c  # initialize residual capacity of the new bin
            s = nb      # select the new bin
            nb += 1     # increase the number of open bins

        bin[i] = s      # pack item i into the selected bin
        rc[s] -= w[i]   # update the residual capacity

    return nb, bin

## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [5]:
# runs every example of the lecture notes and every instance of the exercises,
# and compares the number of bins and the item-bin assignment with the expected ones

errors = 0
checks = 0


def check(instance, algorithm, n, nb, bin, nb_expected, bin_expected):
    global errors, checks
    checks += 1
    ok = (nb == nb_expected)
    for i in range(n):
        if bin[i] != bin_expected[i]:
            ok = False
    if not ok:
        errors += 1
    line = "%-18s %-9s nb=%2d  bin=(" % (instance, algorithm, nb)
    for i in range(n):
        line += "%s%d" % (" " if i else "", bin[i])
    line += ")  %s" % ("OK" if ok else "<-- EXPECTED A DIFFERENT SOLUTION")
    print(line)


algorithms = [("NextFit", next_fit), ("FirstFit", first_fit),
              ("BestFit", best_fit), ("WorstFit", worst_fit)]

# each instance: name, capacity c, weights w, and for each algorithm (in the order above)
# the expected pair (number of bins, bin of each item)
instances = [
    ("Example (c=10)", 10, [6, 5, 4, 7, 3],
     [(3, [0, 1, 1, 2, 2]), (3, [0, 1, 0, 2, 1]), (3, [0, 1, 0, 2, 2]), (3, [0, 1, 1, 2, 0])]),
    ("Half-full NF ok", 10, [5, 5, 5, 5],
     [(2, [0, 0, 1, 1]), (2, [0, 0, 1, 1]), (2, [0, 0, 1, 1]), (2, [0, 0, 1, 1])]),
    ("Half-full FF", 10, [7, 7, 3, 4],
     [(3, [0, 1, 1, 2]), (3, [0, 1, 0, 2]), (3, [0, 1, 0, 2]), (3, [0, 1, 0, 2])]),
    ("NF not Any Fit", 10, [3, 8, 3, 8],
     [(4, [0, 1, 2, 3]), (3, [0, 1, 0, 2]), (3, [0, 1, 0, 2]), (3, [0, 1, 0, 2])]),
    ("Half-full NF bad", 10, [2, 9, 2, 9],
     [(4, [0, 1, 2, 3]), (3, [0, 1, 0, 2]), (3, [0, 1, 0, 2]), (3, [0, 1, 0, 2])]),
    ("EX_*_1", 10, [6, 6, 4, 4],
     [(3, [0, 1, 1, 2]), (2, [0, 1, 0, 1]), (2, [0, 1, 0, 1]), (2, [0, 1, 0, 1])]),
    ("EX_*_2", 10, [7, 7, 3, 3],
     [(3, [0, 1, 1, 2]), (2, [0, 1, 0, 1]), (2, [0, 1, 0, 1]), (2, [0, 1, 0, 1])]),
    ("EX_*_3", 10, [4, 7, 6, 3, 5],
     [(4, [0, 1, 2, 2, 3]), (3, [0, 1, 0, 1, 2]), (3, [0, 1, 0, 1, 2]), (3, [0, 1, 0, 1, 2])]),
    ("worst case n=8", 8, [4, 2, 4, 2, 4, 2, 4, 2],
     [(4, [0, 0, 1, 1, 2, 2, 3, 3]), (3, [0, 0, 1, 0, 1, 2, 2, 2]),
      (3, [0, 0, 1, 0, 1, 2, 2, 2]), (4, [0, 0, 1, 1, 2, 2, 3, 3])]),
    ("worst case n=12", 12, [6, 2, 6, 2, 6, 2, 6, 2, 6, 2, 6, 2],
     [(6, [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5]), (4, [0, 0, 1, 0, 1, 0, 2, 2, 3, 2, 3, 2]),
      (4, [0, 0, 1, 0, 1, 0, 2, 2, 3, 2, 3, 2]), (6, [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5])]),
    ("worst case n=16", 16, [8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2],
     [(8, [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7]),
      (5, [0, 0, 1, 0, 1, 0, 2, 0, 2, 3, 3, 3, 4, 3, 4, 3]),
      (5, [0, 0, 1, 0, 1, 0, 2, 0, 2, 3, 3, 3, 4, 3, 4, 3]),
      (8, [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7])]),
]

for name, c, w, expected in instances:
    n = len(w)
    for a in range(len(algorithms)):
        algorithm, function = algorithms[a]
        nb, bin = function(n, c, w)
        check(name, algorithm, n, nb, bin, expected[a][0], expected[a][1])

print("\n%d checks, %d errors" % (checks, errors))

Example (c=10)     NextFit   nb= 3  bin=(0 1 1 2 2)  OK
Example (c=10)     FirstFit  nb= 3  bin=(0 1 0 2 1)  OK
Example (c=10)     BestFit   nb= 3  bin=(0 1 0 2 2)  OK
Example (c=10)     WorstFit  nb= 3  bin=(0 1 1 2 0)  OK
Half-full NF ok    NextFit   nb= 2  bin=(0 0 1 1)  OK
Half-full NF ok    FirstFit  nb= 2  bin=(0 0 1 1)  OK
Half-full NF ok    BestFit   nb= 2  bin=(0 0 1 1)  OK
Half-full NF ok    WorstFit  nb= 2  bin=(0 0 1 1)  OK
Half-full FF       NextFit   nb= 3  bin=(0 1 1 2)  OK
Half-full FF       FirstFit  nb= 3  bin=(0 1 0 2)  OK
Half-full FF       BestFit   nb= 3  bin=(0 1 0 2)  OK
Half-full FF       WorstFit  nb= 3  bin=(0 1 0 2)  OK
NF not Any Fit     NextFit   nb= 4  bin=(0 1 2 3)  OK
NF not Any Fit     FirstFit  nb= 3  bin=(0 1 0 2)  OK
NF not Any Fit     BestFit   nb= 3  bin=(0 1 0 2)  OK
NF not Any Fit     WorstFit  nb= 3  bin=(0 1 0 2)  OK
Half-full NF bad   NextFit   nb= 4  bin=(0 1 2 3)  OK
Half-full NF bad   FirstFit  nb= 3  bin=(0 1 0 2)  OK
Half-full NF bad   B

## Try it yourself

Change the data in the cell below and run it again.

In [6]:
# Try it yourself: change the data below and run the cell again
c = 10                # bin capacity
w = [6, 5, 4, 7, 3]   # item weights (each at most c)

n = len(w)
print("Instance: c = %d, n = %d items" % (c, n))
for name, function in [("Next Fit", next_fit), ("First Fit", first_fit),
                       ("Best Fit", best_fit), ("Worst Fit", worst_fit)]:
    nb, bin = function(n, c, w)
    print("%-9s : %d bins" % (name, nb))
    for b in range(nb):
        line = "    bin %d: items" % b
        load = 0
        for i in range(n):
            if bin[i] == b:
                line += " %d" % i
                load += w[i]
        print(line + "  (load %d)" % load)

Instance: c = 10, n = 5 items
Next Fit  : 3 bins
    bin 0: items 0  (load 6)
    bin 1: items 1 2  (load 9)
    bin 2: items 3 4  (load 10)
First Fit : 3 bins
    bin 0: items 0 2  (load 10)
    bin 1: items 1 4  (load 8)
    bin 2: items 3  (load 7)
Best Fit  : 3 bins
    bin 0: items 0 2  (load 10)
    bin 1: items 1  (load 5)
    bin 2: items 3 4  (load 10)
Worst Fit : 3 bins
    bin 0: items 0 4  (load 9)
    bin 1: items 1 2  (load 9)
    bin 2: items 3  (load 7)


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).